### Import packages
#### Create connection to Database

In [19]:
from sqlalchemy import create_engine
import pandas as pd

# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

#### Read table from database

In [6]:
# HINTS
# Load data into DataFrame
# user Pandas to read data from a table into a DataFrame

In [21]:
# # Approach 1: Use Pandas.read_sql_table to read all columns from 'customers' table
table_name = 'customers'
df = pd.read_sql_table(table_name, db_engine)
df.tail(5)

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
54,55,Mark,Taylor,None,421 Bourke Street,Sidney,NSW,Australia,2010,+61 (02) 9332 3633,None,mark.taylor@yahoo.au,4
55,56,Diego,Gutiérrez,None,307 Macacha Güemes,Buenos Aires,None,Argentina,1106,+54 (0)11 4311 4333,None,diego.gutierrez@yahoo.ar,4
56,57,Luis,Rojas,None,"Calle Lira, 198",Santiago,None,Chile,None,+56 (0)2 635 4444,None,luisrojas@yahoo.cl,5
57,58,Manoj,Pareek,None,"12,Community Centre",Delhi,None,India,110017,+91 0124 39883988,None,manoj.pareek@rediff.com,3
58,59,Puja,Srivastava,None,"3,Raj Bhavan Road",Bangalore,None,India,560001,+91 080 22289999,None,puja_srivastava@yahoo.in,3


In [8]:
# # Approach 2: Use Pandas.read_sql_query to read these columns
# table_name = 'customers'
# columns = ['CustomerId', 'FirstName', 'LastName', 'Phone', 'Email', 'SupportRepId']
df = pd.read_sql_query(sql= 'select customerid, firstname from customers', con= db_conn)
df.tail(5)

,CustomerId,FirstName
54,55,Mark
55,56,Diego
56,57,Luis
57,58,Manoj
58,59,Puja


Homework 1. Metadata-driven approach. Dung Query SQLite system tables directly (No SQLAlchemy inspector)

In [53]:
# list all table names in the database chinook.db
# df['column_name'] accesses a column . It returns a pandas Series , A Series can be converted to a Python list using .tolist()
# can do chaining df = pd.read_sql_query(...)['name'].tolist()

df = pd.read_sql_query(
    "Select name From sqlite_master Where type ='table';",
    con = db_conn
)

table_names = df['name'].tolist()
print(table_names)
print('end cach 1')


#cach 2: su dung inspect

from sqlalchemy import create_engine, inspect

engine = create_engine('sqlite:///chinook.db')
inspector = inspect(engine)

tables2 = inspector.get_table_names()
print(tables2)

['albums', 'sqlite_sequence', 'artists', 'customers', 'employees', 'genres', 'invoices', 'invoice_items', 'media_types', 'playlists', 'playlist_track', 'tracks', 'sqlite_stat1']
end cach 1
['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']


In [ ]:
# loads each table into a dataframe
# dfs ={} create an empty dictionary, and will be filled in with key = table name, and value = dataframe for that table

dfs = {}
for table in table_names:
    dfs[table] = pd.read_sql_query(f'Select * from {table};', con=db_conn)

print(dfs)

In [11]:
# load all the tables to csv files, save in destination folder newly created 
# have to do in dfs.item() because it gave both key and value, if only dfs, it will give the key only 
import os 

os.makedirs('destination', exist_ok= True)
for name, df in dfs.items():
    df.to_csv(f'destination/{name}.csv', index=False)

#### Config-Driven Ingestion

In [44]:
# # HINTS
# # Read configs stored in the 'config.yml' file

# # Read yaml file
# # Package: yaml (pip install pyyaml)
# # Function: load / safe_load
# # Print it after loading

# import yaml
# import json

# config_file = 'config.yml'

In [ ]:
#config['table'] ❌ → KeyError
#config['source']['table'] ✅ → the list of table names boi vi table nam ben duoi source: mo file config.yml ra xem 

import yaml
import pandas as pd

with open('config.yml') as f:
    config = yaml.safe_load(f)
engine = create_engine('sqlite:///chinook.db')

dfs = {}

for table in config['source']['table']:
    dfs[table] = pd.read_sql_table(table, engine)

print(dfs)

In [46]:
# Use loop function to read tables within config.source.table
# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: destination/config_driven/
# note: use os.makedirs() if path is not exists


In [ ]:
import os
import datetime
import yaml
import pandas as pd

path = 'destination/config_date'
engine = create_engine('sqlite:///chinook.db')
date = datetime.date.today().strftime('%d%m%y')

os.makedirs(path, exist_ok=True)

with open('config.yml') as f:
    config = yaml.safe_load(f)

for table in config['source']['table']:
    df = pd.read_sql_table(table,engine)
    
    file_name = f'{date}_{table}.csv'
    full_path = os.path.join(path,file_name)

    df.to_csv(full_path,index=False)

    print(f'Saved: {full_path}')

# trong phan nay ko can dung dfs{} vi neu xuat ra csv file thi ko can. chi dung dfs{} khi can storing de lam viec voi table do later


Same logic, nhung viet lai duoi dang resuable functions

In [ ]:
from sqlalchemy import create_engine
import yaml, os, datetime
import pandas as pd

with open('config.yml') as f:
    config = yaml.safe_load(f)



def create_connection (config_dict):
    db_type = config_dict['source']['database']['db_type']
    host = config_dict['source']['database']['host']

    if db_type == 'sqlite':
        conn_str = f'sqlite:///{host}.db'
    else:
        raise NotImplementedError('Only Sqlite supported for now')
    
    return create_engine(conn_str)

def extract_table (table_name, engine, output_path, date_str):
    os.makedirs(output_path, exist_ok=True)

    df = pd.read_sql_table(table_name, engine)

    file_name = f'{date_str}_{table_name}.csv'
    full_path = os.path.join(output_path,file_name)
    df.to_csv(full_path,index=False)
    print(f'Complete {file_name}')

engine = create_connection(config)

date_str = datetime.date.today().strftime('%d.%m.%y')
output_path = 'destination/config_function'

for table_name in config['source']['table']:
    extract_table(
        table_name=table_name,
        engine=engine,
        output_path= output_path,
        date_str=date_str
    )

In [49]:
# # HINTS
# # Read metadata from the database inlcuding tables / columns
# sqlite_metadata_table = 'sqlite_master'
# sqlite_metadata_condition = "type = 'table'"
# metadata_sql = f""" select 1"""
# print(metadata_sql)
# table_df = pd.read_sql_query(metadata_sql)
# print(table_df)

In [50]:
# loop for each table from the DataFrame
# read and extract table
# save to path: destination/metadata_driven/
# note: use os.makedirs() if path is not exists